# Predictive analysis of Bank Marketing

#### Problem Statement
The data is related with direct marketing campaigns of a Portuguese banking institution. The marketing campaigns were based on phone calls. Often, more than one contact to the same client was required, in order to access if the product (bank term deposit) would be ('yes') or not ('no') subscribed. 

#### What to achieve?
The classification goal is to predict if the client will subscribe a term deposit (variable y).

#### Data Contains information in following format:

### Categorical Variable :

* Marital - (Married , Single , Divorced)",
* Job - (Management,BlueCollar,Technician,entrepreneur,retired,admin.,services,selfemployed,housemaid,student,unemployed,unknown)
* Contact - (Telephone,Cellular,Unknown)
* Education - (Primary,Secondary,Tertiary,Unknown)
* Month - (Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec)
* Poutcome - (Success,Failure,Other,Unknown)
* Housing - (Yes/No)
* Loan - (Yes/No)
* Default - (Yes/No)

### Numerical Variable:

* Age
* Balance
* Day
* Duration
* Campaign
* Pdays
* Previous

#### Class
* deposit - (Yes/No)

In [ ]:
#Importing required libraries
import pandas as pd
import numpy as np
# from sklearn.cross_validation import train_test_split
from sklearn.model_selection import train_test_split
from sklearn import metrics
import statsmodels.formula.api as smf
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder

# visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

#creating HDR slices
from sklearn.neighbors import KernelDensity
import arviz as az
from sklearn import metrics

# data synthesizing
from collections import Counter
from ctgan import CTGANSynthesizer
from ctgan import load_demo

# synthesizing data with ADASYN
neighbs = 5 # number of neighbours used to construct synthetic data
scale = 1.5

## Hyper-parameters

In [ ]:
# splitting dataset
val_size = .15
test_size = .15
train_size = 1 - val_size - test_size

# creating hdr slices
hdi_prob = 0.9

# importance of slices
sig = .02
min_support = .05

# synthesizing data
gen_p = .75
num_problem_slices = 3

## Exploring dataset

In [ ]:
# #Importing and displaying data
data = pd.read_csv("bank-full.csv", delimiter=";",header='infer')
data.head()

#Importing and displaying data
# data = pd.read_csv("german.data", sep=" ", header=None)
# data.head()

In [ ]:
#4521 rows and 17 features
data.shape

In [ ]:
#datatypes of the columns
data.dtypes

In [ ]:
#Since y is a class variable we will have to convert it into binary format. (Since 2 unique class values)
data.y.replace(('yes', 'no'), (1, 0), inplace=True)

# data = data.rename(columns={20:'y'})

# data.y.replace((1, 2), (1, 0), inplace=True)

In [ ]:
#Spliting data as X -> features and y -> class variable
data_y = pd.DataFrame(data['y'])
data_X = data.drop(['y'], axis=1)
print(data_X.columns)
print(data_y.columns)

## Spliting and manipulating datasets

In [ ]:
#Dividing records in training validation and testing sets along with its shape (rows, cols)
X_train, X_test_raw, y_train, y_test = \
train_test_split(data_X, data_y, test_size=test_size, random_state=2, stratify=data_y)

X_train_raw, X_val_raw, y_train, y_val = \
train_test_split(X_train, y_train, test_size=val_size/(val_size+train_size), random_state=2, stratify=y_train)

print (X_train_raw.shape)
print(X_val_raw.shape)
print (X_test_raw.shape)

print (y_train.shape)
print(y_val.shape)
print (y_test.shape)

Since the dtype contains types other than int, float; we need to convert those column values into proper format in order to fit the data in model.

In [ ]:
cat_feats = []
for i, col in enumerate(data.dtypes):
    if np.issubdtype(col, np.integer) == False and np.issubdtype(col, np.floating) == False:
        if data.columns[i] == 'y':
            pass
        else:
            cat_feats.append(data.columns[i])

cat_feats

In [ ]:
#Converting object type data into numeric type using One-Hot encoding method which is
#majorly used for XGBoost (for better accuracy) [Applicable only for non numeric categorical features]
X_train = pd.get_dummies(X_train_raw, columns=cat_feats)
X_val = pd.get_dummies(X_val_raw, columns=cat_feats)
X_test = pd.get_dummies(X_test_raw, columns=cat_feats)
#pd is instance of pandas. Using get_dummies method we can directly convert any type of data into One-Hot encoded format.

In [ ]:
#Checking types of all the columns converted
X_train.dtypes

In [ ]:
for col in X_train.columns:
    if col not in X_val:
        X_val.insert(loc=X_train.columns.get_loc(col), column=col,value=0)
    if col not in X_test:
        X_test.insert(loc=X_train.columns.get_loc(col), column=col,value=0)

## Regular Training

In [ ]:
#Create an XGB classifier and train it on 70% of the data set.
from sklearn import svm
from xgboost import XGBClassifier
clf = XGBClassifier()
clf

In [ ]:
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_val)
val_acc = np.sum(y_pred.reshape([-1,1]) == np.array(y_val)) / len(y_pred)
val_acc

In [ ]:
baseline = metrics.f1_score(y_val, y_pred)

In [ ]:
#classification accuracy
from sklearn import metrics
print(metrics.accuracy_score(y_val, y_pred))

# Improving Model

## Creating HDR Slices

In [ ]:
def f1(TP,TN,FP,FN):
    return TP/(TP+0.5*(FN+FP)) 

def recall(TP,TN,FP,FN):
    return TP/(TP+FN) 

def precision(TP,TN,FP,FN):
    return TP/(TP+FP) 

def f_beta(TP,TN,FP,FN, beta):
    return ((1+beta*beta)*TP)/((1+beta*beta)*TP+beta*beta*FN+0.5*FP) 

def f_half(TP,TN,FP,FN):
    return f_beta(TP,TN,FP,FN,0.5)

In [ ]:
def confusion_metrics(row):
    if row['y'] == 1 and row['y_pred'] == 0:
        val = 'FN'
    elif row['y'] == 0 and row['y_pred'] == 1:
        val = 'FP'
    elif row['y'] == 1:
        val = 'TP'
    else:
        val = 'TN'

    return val

In [ ]:
# generalized function for comparing accuracy
def compare_acc(dfx, dfy, y_pred, sl):
    feat = sl[0][0]
    if sl[0][1] is None:
        indexes = np.arange(dfx.index[:][-1])
    elif type(sl[0][1]) == list:
        hdr_min, hdr_max = sl[0][1][0], sl[0][1][1]
        if hdr_min == hdr_max:
            indexes = dfx.index[dfx[feat] == hdr_min]
        else:
            indexes = dfx.index[dfx[feat].between(hdr_min, hdr_max)]
    else:
        indexes = dfx.index[dfx[feat] == sl[0][1]]
    test_acc = np.sum(y_pred == np.array(dfy.values)[:,0]) / len(y_pred)
    sub_acc = np.sum(y_pred[indexes] == np.array(dfy.loc[indexes].values)[:,0]) / len(indexes)

    dif = test_acc - sub_acc
    
    return dif, sub_acc

In [ ]:
def compare_f1(dfx, y, y_pred , sl):
    feat = sl[0][0]
    
    val_total_f1 = metrics.f1_score(y, y_pred)
    
    main = pd.DataFrame()
    main[feat] = dfx[feat].copy()

    main['y_pred'] = y_pred
    main['y'] = y
    
    main['metrics'] = main.apply(confusion_metrics, axis=1)

    if type(sl[0][1]) == list:
        left = sl[0][1][0]
        right = sl[0][1][1]

        indexes = dfx.index[dfx[feat].between(left, right)]
        new_dfx = main.loc[indexes]
        new_y = y.loc[indexes]
        new_y_pred = y_pred[indexes]
        
    else:
        val = sl[0][1]
        
        indexes = dfx.index[dfx[feat] == val]
        new_dfx = main.loc[indexes]
        new_y = y.loc[indexes]
        new_y_pred = y_pred[indexes]
    
    TP = new_dfx[feat].where(new_dfx.metrics == 'TP').count()
    TN = new_dfx[feat].where(new_dfx.metrics == 'TN').count()
    FP = new_dfx[feat].where(new_dfx.metrics == 'FP').count()
    FN = new_dfx[feat].where(new_dfx.metrics == 'FN').count()
    if FP == 0 and FN == 0:
        val_slice_f1 = 1
    else:
        
        val_slice_f1 = metrics.f1_score(new_y, new_y_pred)

    dif = val_total_f1 - val_slice_f1

    return dif, val_slice_f1

In [ ]:
def find_range(data,y,y_pred ,col, hdi_prob = hdi_prob, q1=2,q2 = 5, func = f1, baseline = baseline): # with moving window
    # HDR
    #   print(f"Checking {col} feature:")

    main = pd.DataFrame()
    main[col] = data[col].copy()

    main['y_pred'] = y_pred
    main['y'] = y

    main['metrics'] = main.apply(confusion_metrics, axis=1)


    np_data = main[col].to_numpy()
    hdr = az.hdi(np_data, hdi_prob=hdi_prob)

    new_left = hdr[0]
    new_right = hdr[1]

    #print(f"HDR = [{new_left}, {new_right}]")


    new_data = main.where(main[col] >= new_left).where(main[col] <= new_right)

    TP = new_data[col].where(new_data.metrics == 'TP').count() # always closed from right
    TN = new_data[col].where(new_data.metrics == 'TN').count()
    FP = new_data[col].where(new_data.metrics == 'FP').count()
    FN = new_data[col].where(new_data.metrics == 'FN').count()
    acc = func(TP,TN,FP,FN)

    #print(f"HDR accuracy measure= {acc:.3f}")


    # find bigest error in HDR using pd.qcut
    slices = []
    lowest_acc = 1
    slice_to_upgrade = None
    slices_under_baseline = []
    worst_matrix = [0,0,0,0]
    n_unique = len(pd.unique(new_data[col]))

    #print(n_unique)
    if q2> n_unique/2:
        q2 = n_unique

    for q in range(q1,q2+1):
        x = pd.qcut(new_data[col], q=q, duplicates='drop').value_counts()
        #x = pd.cut(new_data, bins=q, duplicates='drop').value_counts()
        if len(x) < q:
          #print(f'not splittiable with qcut to {q} intervals')
          break
    
    for interval in x.index:
      left = interval.left
      right = interval.right
      TP = new_data[col].where(new_data.metrics == 'TP').where(new_data[col] > left).where(new_data[col] <= right).count() # always closed from right
      TN = new_data[col].where(new_data.metrics == 'TN').where(new_data[col] > left).where(new_data[col] <= right).count()
      FP = new_data[col].where(new_data.metrics == 'FP').where(new_data[col] > left).where(new_data[col] <= right).count()
      FN = new_data[col].where(new_data.metrics == 'FN').where(new_data[col] > left).where(new_data[col] <= right).count()
      acc = func(TP,TN,FP,FN)

      if  acc < lowest_acc: #and acc > 0:
        lowest_acc = acc
        slice_to_upgrade = interval
        worst_matrix = [TP, TN, FP, FN]

      slices.append([interval, acc, TP, TN, FP, FN])
      if acc < baseline:
        slices_under_baseline.append(interval)

  #print(f"worst slice = {slice_to_upgrade} with accuracy of {lowest_acc}")
  #print(f"(TP,TN,FP,FN) = ({worst_matrix[0]},{worst_matrix[1]},{worst_matrix[2]},{worst_matrix[3]})")

    return slices_under_baseline, slice_to_upgrade, lowest_acc, worst_matrix

In [ ]:
# get list of all numeric feature slices
num_feats = []
for i, col in enumerate(data.dtypes):
    if np.issubdtype(col, np.integer):
        if data.columns[i] == 'y':
            pass
        else:
            num_feats.append(data.columns[i])

num_feats

In [ ]:
df_x = X_val.reset_index(drop=True)
df_y = y_val.reset_index(drop=True)

compare = y_pred == np.array(df_y.values[:,0])
bad_guesses = X_val.iloc[np.where(compare == 0)[0]]

hdr_dict = {}
for feat in num_feats:
    _, slice_to_upgrade, _, _ = find_range(df_x,df_y,y_pred ,col = feat)
    if slice_to_upgrade == None:
        pass
    else:
        hdr_dict[feat] = [slice_to_upgrade.left + .0001, slice_to_upgrade.right]

hdr_dict

## Creating All Slices

In [ ]:
from itertools import combinations
# generate slices
feats_and_vals = []
for feat in X_train.columns:
    # for numeric features only create slices in hdr
    if feat in num_feats:
        if feat not in hdr_dict.keys():
            pass
        else:
            feats_and_vals.append([feat, hdr_dict[feat]])
    # for categorical features, create slices for each unique value
    else:
        for val in X_train[feat].unique():
            if val == 0:
                pass
            else:
                feats_and_vals.append([feat, val])

combos = list(combinations(feats_and_vals,1))
combos

## Filtering Problematic Slices

In [ ]:
def test_error(prob_slice):
    feat = prob_slice[0][0]
    
    if type(prob_slice[0][1]) == list:
        hdr_min, hdr_max = prob_slice[0][1][0], prob_slice[0][1][1]
        if hdr_min == hdr_max:
            test_error = len(bad_guesses[bad_guesses[feat] == hdr_min]) / len(bad_guesses)
        else:
            test_error = len(bad_guesses[bad_guesses[feat].between(hdr_min, hdr_max)]) / len(bad_guesses)
    else:
        val = prob_slice[0][1]
        test_error = len(bad_guesses[bad_guesses[feat] == val]) / len(bad_guesses)
    
    return test_error

In [ ]:
# check importance of slices
prob_slices = []
for sl in combos:
    dif, acc = compare_f1(df_x, df_y, y_pred, sl)
    if dif > sig:
        error = test_error(sl)
        if error >= min_support:
            prob_slices.append([sl, acc, error])

prob_slices

## Adding oversampled data to training set

In [ ]:
# scoring importance of problem slices and extracting the most important
num_slices = num_problem_slices
slices_score = []
for sl in prob_slices:
    feat = sl[0][0][0]
    error, f1 = sl[2],sl[1]
    if f1==0:
        f1=.1
    score = error/f1
    slices_score.append(score)
    
slices_score = np.array(slices_score) * (-1)
top = list(slices_score.argsort()[:num_slices])

important_slices = []
for idx in top:
    important_slices.append(prob_slices[idx])

important_feats = []
for sl in important_slices:
    feat = sl[0][0][0]
    important_feats.append(feat)

In [ ]:
# creating new dataset with oversampled important slices

X = X_train.copy().reset_index(drop=True)
y = y_train.copy().reset_index(drop=True)

X_train_new = X_train.copy().reset_index(drop=True)
y_train_new = y_train.copy().reset_index(drop=True)



for prob_slice in important_slices:
    column_name = prob_slice[0][0][0]
    key = column_name

    is_categorical = '_' in column_name

    if is_categorical:
        column_value = 1
        
        rand = X[X[column_name] == column_value]
        rand_y = y.loc[rand.index]

    else:
        prob_feat = prob_slice
        column_from_val, column_to_val = prob_slice[0][0][1]
        
        rand = X[X[column_name].between(column_from_val, column_to_val)]
        rand_y = y.loc[rand.index]
        
        
    ####
    minority_class = np.argmin(rand_y.value_counts())
    minority_idx = rand_y[rand_y['y'] == minority_class].index
    majority_idx = rand_y[rand_y['y'] != minority_class].index

    X_minority = rand.copy()
    X_minority = X_minority.loc[minority_idx]
    y_minority = rand_y.loc[minority_idx]

    X_majority = rand.copy()
    X_majority = X_majority.loc[majority_idx]
    y_majority = rand_y.loc[majority_idx]
    ####

    N = int(len(X_minority) * gen_p)
    rand_minority = X_minority.sample(n=N)
    rand_y_minority = y_minority.loc[rand_minority.index]
    rand_majority = X_majority.sample(n=N)
    rand_y_majority = y_majority.loc[rand_majority.index]
    min_class = np.argmin(y.value_counts())
    min_idx = y[y['y'] == min_class].index
    X_rand = X.loc[min_idx].sample(n=N)
#     X_rand = X.sample(n=N)
    y_rand = y.loc[X_rand.index]

    X_train_new = X.copy().append(rand_minority)
    X_train_new = X_train_new.append(rand_majority)
    
    y_train_new = y.copy().append(rand_y_minority)
    y_train_new = y_train_new.append(rand_y_majority)
    
    X_train_new_2 = X.copy().append(rand_minority)
    
    y_train_new_2 = y.copy().append(rand_y_minority)
    
    X_train_random = X.copy().append(X_rand)
    y_train_random = y.copy().append(y_rand)

In [ ]:
important_slices

In [ ]:
f1s_val = {}
for i, sl in enumerate(important_slices):
    sl_name = sl[0][0][0]
    dif, f1 = compare_f1(df_x, df_y, y_pred , sl[0])
    if i == 0:        
        f1s_val['total_acc'] = val_acc
        f1s_val['total_f1'] = f1 + dif
    f1s_val[f'{sl_name}_f1'] = f1

## Adding Synthetic Data to Training Set

In [ ]:
# Generating new data for all problem slices

from imblearn.over_sampling import ADASYN
from collections import Counter

def synthesize(prob_slice, X, y, cat_feats, scale):
    # extract info from prob slice
    prob = prob_slice
    
    prob_slice = prob[0][0][0]
    cat = '_' in prob_slice
    if cat:
        prob_slice = prob_slice.split('_')

        prob_feat, prob_val = prob_slice[0], prob_slice[1]
        
    else:
        prob_feat = prob_slice
        left, right = prob[0][0][1]
        

    # factorize dataset 
    data_fact = X.copy(deep=True)
    data_fact['y'] = y

    unique_dict = {}
    for feat in cat_feats:
        data_fact[feat], unique_dict[feat] = pd.factorize(data_fact[feat])
        
    if cat:
        # identify factorized values of prob_val
        prob_fact_val = unique_dict[prob_feat].get_loc(prob_val)
#         print(prob_feat)
#         print(prob_val, prob_fact_val)
#         print(unique_dict[prob_feat])
    
    # initialize synthesize function
    ada = ADASYN(random_state=42, n_neighbors=neighbs)
    if cat:
        sl_df = data_fact[data_fact[prob_feat] == prob_fact_val]
        sl_size = len(sl_df)

        p = sl_size / len(data_fact)
        compare_p = scale * p
        compare_size = int(len(data_fact) * compare_p)
        
        filter_df = data_fact[data_fact[prob_feat] != prob_fact_val]
#         print(prob_feat, len(filter_df), len(X[X[prob_feat] != prob_val]))
        if len(filter_df) > compare_size:
            filter_df = filter_df.sample(n=compare_size)
        if len(filter_df) < len(sl_df):
            synth_X, synth_y = None, None    
            return synth_X, synth_y
#         print(len(filter_df), compare_size, sl_size)    
        data_ret = sl_df.append(filter_df)
        
        data_ret[prob_feat] = np.where(data_ret[prob_feat] == prob_fact_val, 1, 0)

    else:
        sl_df = data_fact[data_fact[prob_feat].between(left, right)]
        sl_size = len(sl_df)

        p = sl_size / len(data_fact)
        compare_p = scale * p
        compare_size = int(len(data_fact) * compare_p)

        filter_df = data_fact[~data_fact[prob_feat].between(left, right)]
        if len(filter_df) > compare_size:
            filter_df = filter_df.sample(n=compare_size)
        if len(filter_df) < len(sl_df):
            synth_X, synth_y = None, None    
            return synth_X, synth_y
        
        data_ret = sl_df.append(filter_df)

        data_ret[f'actual_{prob_feat}'] = data_ret[prob_feat]
        data_ret[prob_feat] = np.where(data_ret[prob_feat].between(left,right), 'in', data_ret[prob_feat])
        data_ret[prob_feat] = np.where(data_ret[prob_feat] != 'in', 'out', data_ret[prob_feat])
        data_ret[prob_feat] = np.where(data_ret[prob_feat] == 'in', 1, data_ret[prob_feat])
        data_ret[prob_feat] = np.where(data_ret[prob_feat] != 1, 0, data_ret[prob_feat])
        
    # synthesize data after switching y to prob_feat
    y_train = data_ret[prob_feat]
    x_train = data_ret.drop([prob_feat],axis=1)
    y_train=y_train.astype('int')
#     print(prob_feat, compare_size, len(data_fact))
#     print(y_train.value_counts())
    X_train_ada_feat, y_train_ada_feat = ada.fit_resample(x_train, y_train)
    
    # add prob_feat back to X in appropriate column
    idx = X.columns.get_loc(prob_feat)
    df_ada = X_train_ada_feat.copy()
    if cat:
        df_ada.insert(idx, prob_feat, y_train_ada_feat)
    else:
        df_ada = df_ada.drop(f'actual_{prob_feat}', axis=1)
        df_ada.insert(idx, prob_feat, X_train_ada_feat[f'actual_{prob_feat}'])

    # convert factorized values back to categorical values
    synth_df = df_ada.copy()
    for feat in cat_feats:
        if feat == prob_feat:
            synth_df[feat] = np.where(synth_df[feat] == 1, prob_val, 'whats going on')
        else:    
            synth_df[feat] = unique_dict[feat][synth_df[feat]]

    # seperate real y from X
    synth_y = synth_df['y']
    synth_X = synth_df.drop('y', axis=1)
        

    N = len(x_train)

    # exract only synthetic data from dataset
    synth_X = synth_X.iloc[N:]
    synth_y = synth_y.iloc[N:]
    
    return synth_X, synth_y

In [ ]:
# Generating new data for all problem slices

X, y = X_train_raw.copy(), y_train.copy()
cat_feats = cat_feats

synth_data = {}
for sl in prob_slices:
    synth_data[f'{sl}'] = synthesize(sl, X, y, cat_feats, scale)
    
# Adding synthesized data to training set

X_train_synth, y_train_synth = X_train_raw.copy(), y_train.copy()

for dfs in synth_data.values():
    if dfs[0] is None:
        pass
    else:
        X_train_synth = X_train_synth.append(dfs[0])
        y_train_synth = y_train_synth.append(dfs[1].to_frame())
        
X_train_synth = pd.get_dummies(X_train_synth, columns=cat_feats)

## Regular Boosted Model Performance on Test Set

In [ ]:
y_pred_test = clf.predict(X_test)
np.array(y_pred).shape
test_acc = np.sum(y_pred_test.reshape([-1,1]) == np.array(y_test)) / len(y_pred_test)
test_acc

In [ ]:
df_x_test = X_test.reset_index(drop=True)
df_y_test = y_test.reset_index(drop=True)

idx0 = df_y_test[df_y_test['y'] == 0].index
idx1 = df_y_test[df_y_test['y'] == 1].index

x0 = df_x_test.loc[idx0]
y0 = df_y_test.loc[idx0]
y_pred0 = y_pred_test[idx0]

x1 = df_x_test.loc[idx1]
y1 = df_y_test.loc[idx1]
y_pred1 = y_pred_test[idx1]

In [ ]:
metrics.accuracy_score(df_y_test, y_pred_test), \
metrics.recall_score(df_y_test, y_pred_test, pos_label=0), \
metrics.precision_score(df_y_test, y_pred_test, pos_label=0), \
metrics.f1_score(df_y_test, y_pred_test, pos_label=0)

In [ ]:
metrics.accuracy_score(df_y_test, y_pred_test), \
metrics.recall_score(df_y_test, y_pred_test, pos_label=1), \
metrics.precision_score(df_y_test, y_pred_test, pos_label=1), \
metrics.f1_score(df_y_test, y_pred_test, pos_label=1)

In [ ]:
df_x_test = X_test.reset_index(drop=True)
df_y_test = y_test.reset_index(drop=True)

f1s_test = {}
for i, sl in enumerate(important_slices):
    sl_name = sl[0][0][0]
    dif, f1 = compare_f1(df_x_test, df_y_test, y_pred_test , sl[0])
    if i == 0:        
        f1s_test['total_acc'] = test_acc
        f1s_test['total_f1'] = f1 + dif
    f1s_test[f'{sl_name}_f1'] = f1

## Updated Both Classes Synthetic Performance on Test Set

In [ ]:
clf.fit(X_train_synth, y_train_synth)

y_pred_test_new = clf.predict(X_test)
test_acc_new = np.sum(y_pred_test_new.reshape([-1,1]) == np.array(y_test)) / len(y_pred_test_new)
test_acc_new

In [ ]:
f1s_test_synth = {}
for i, sl in enumerate(important_slices):
    sl_name = sl[0][0][0]
    dif, f1 = compare_f1(df_x_test, df_y_test, y_pred_test_new , sl[0])
    if i == 0:        
        f1s_test_synth['total_acc'] = test_acc_new
        f1s_test_synth['total_f1'] = f1 + dif
    f1s_test_synth[f'{sl_name}_f1'] = f1

## Updated Random Resampling Performance on Test Set

In [ ]:
clf.fit(X_train_random, y_train_random)

y_pred_test_new = clf.predict(X_test)
test_acc_new = np.sum(y_pred_test_new.reshape([-1,1]) == np.array(y_test)) / len(y_pred_test_new)
test_acc_new

In [ ]:
f1s_test_random = {}
for i, sl in enumerate(important_slices):
    sl_name = sl[0][0][0]
    dif, f1 = compare_f1(df_x_test, df_y_test, y_pred_test_new , sl[0])
    if i == 0:        
        f1s_test_random['total_acc'] = test_acc_new
        f1s_test_random['total_f1'] = f1 + dif
    f1s_test_random[f'{sl_name}_f1'] = f1

## Updated Minority Class Resampling Performance on Test Set

In [ ]:
clf.fit(X_train_new_2, y_train_new_2)

y_pred_test_new = clf.predict(X_test)
test_acc_new = np.sum(y_pred_test_new.reshape([-1,1]) == np.array(y_test)) / len(y_pred_test_new)
test_acc_new

In [ ]:
f1s_test_minority = {}
for i, sl in enumerate(important_slices):
    sl_name = sl[0][0][0]
    dif, f1 = compare_f1(df_x_test, df_y_test, y_pred_test_new , sl[0])
    if i == 0:        
        f1s_test_minority['total_acc'] = test_acc_new
        f1s_test_minority['total_f1'] = f1 + dif
    f1s_test_minority[f'{sl_name}_f1'] = f1

## Updated Both Classes Resampling Performance on Test Set

In [ ]:
clf.fit(X_train_new, y_train_new)

y_pred_test_new = clf.predict(X_test)
test_acc_new = np.sum(y_pred_test_new.reshape([-1,1]) == np.array(y_test)) / len(y_pred_test_new)
test_acc_new

In [ ]:
f1s_test_oversampled = {}
for i, sl in enumerate(important_slices):
    sl_name = sl[0][0][0]
    dif, f1 = compare_f1(df_x_test, df_y_test, y_pred_test_new , sl[0])
    if i == 0:        
        f1s_test_oversampled['total_acc'] = test_acc_new
        f1s_test_oversampled['total_f1'] = f1 + dif
    f1s_test_oversampled[f'{sl_name}_f1'] = f1

## Conclusion from attempted improvements

In [ ]:
f1s_df = pd.DataFrame()
f1s_df['measure'] = f1s_val.keys()
# f1s_df['validation_f1'] = f1s_val.values()
f1s_df['Base'] = f1s_test.values()
f1s_df['Oversampled Synthetic'] = f1s_test_synth.values()
f1s_df['Oversampled Random'] = f1s_test_random.values()
f1s_df['Oversampled Minority'] = f1s_test_minority.values()
f1s_df['Oversampled Balanced'] = f1s_test_oversampled.values()

In [ ]:
f1s_df